# Scouting and Draft Prediction

Two models: a classifier for whether a player-season leads to being drafted, and
a regressor for where a drafted player falls within their college class.

Both are trained only on public NCAA statistics and this package's own derived
metrics. **Read `model_card()` before quoting any of these numbers** — it
carries the limitations.

Needs the `model` extra (`pip install "ncaa_bbStats[model]"`); explanations use
`explain` for SHAP, falling back to gain-based attribution otherwise.

Covers: `scouting_report`, `predict_draft_probability`, `predict_draft_order`,
`draft_board`, `explain_prediction`, `predict_from_stats`, `is_draft_eligible`,
`model_card`

In [1]:
from ncaa_bbStats import *

## A scouting report

In [2]:
print(scouting_report("Kade Anderson", 2025))

  Kade Anderson  |  2025  |  pitcher
  LSU  (SEC)
  Draft grade: A+   (modelled probability 98.5%)
  Projected college draft order: ~1
  Draft eligible: True (basis: drafted)
  Actual: selected #3 in round 1
------------------------------------------------------------------
  Take: the model sees a top-of-the-draft profile. Driven by age.
  Main concern: BB (Batting) (team).

  Top 5 strengths
    feature                            value      median    impact
    ^ age                             20.000      22.000     5.92%
    ^ so (pitching)                  180.000      22.000     5.12%
    ^ k% (pitching)                    0.374       0.189     1.04%
    ^ ip per g (pitching)              6.263       1.556     1.03%
    ^ k-bb% (pitching)                 0.301       0.075     0.63%

  Top 5 concerns
    feature                            value      median    impact
    v BB (Batting) (team)            360.000     247.000    -0.21%
    v DPPG (team)                      0.510     

## The individual predictions behind it

In [3]:
for name in ["Kade Anderson", "Jamie Arnold", "Gavin Kilen", "Jac Caglianone"]:
    probability = predict_draft_probability(name, 2025)
    if probability is None:
        print(f"  {name:22s} not in the eligible population")
        continue
    order = predict_draft_order(name, 2025)
    print(f"  {name:22s} P(drafted) {probability:.1%}   "
          f"projected college order ~{order:.0f}")

  Kade Anderson          P(drafted) 98.5%   projected college order ~1
  Jamie Arnold           P(drafted) 99.5%   projected college order ~2
  Gavin Kilen            P(drafted) 98.6%   projected college order ~17
  Jac Caglianone         not in the eligible population


### Eligibility is inferred, not looked up

It comes from seasons completed and from age — and age is itself estimated for
most players. So the basis is returned alongside the answer.

In [4]:
for name in ["Kade Anderson", "Jamie Arnold", "Jac Caglianone"]:
    result = is_draft_eligible(name, 2025)
    if result is None:
        # Caglianone was drafted in 2024, so he has no 2025 college season.
        print(f"  {name:22s} no 2025 season in the data")
        continue
    eligible, basis = result
    print(f"  {name:22s} eligible={eligible}  basis={basis!r}")

  Kade Anderson          eligible=True  basis='drafted'
  Jamie Arnold           eligible=True  basis='drafted'
  Jac Caglianone         no 2025 season in the data


## What drove the prediction

SHAP where installed, gain-weighted deviation from the median otherwise. Impact
is in percentage points of draft probability.

In [5]:
explanation = explain_prediction("Kade Anderson", 2025, top_n=6)
print(f"method: {explanation['method']}")
print(f"draft probability: {explanation['draft_probability']:.1%}\n")

print("strengths:")
for row in explanation["strengths"]:
    print(f"  ^ {row['label']:28s} {row['value']:>10.3f} "
          f"(median {row['median']:>8.3f})  {row['impact']:+.2f}%")

print("\nconcerns:")
for row in explanation["concerns"]:
    print(f"  v {row['label']:28s} {row['value']:>10.3f} "
          f"(median {row['median']:>8.3f})  {row['impact']:+.2f}%")

method: shap
draft probability: 98.5%

strengths:
  ^ age                              20.000 (median   22.000)  +5.92%
  ^ so (pitching)                   180.000 (median   22.000)  +5.12%
  ^ k% (pitching)                     0.374 (median    0.189)  +1.04%
  ^ ip per g (pitching)               6.263 (median    1.556)  +1.03%
  ^ k-bb% (pitching)                  0.301 (median    0.075)  +0.63%
  ^ start share (pitching)            1.000 (median    0.062)  +0.53%

concerns:
  v BB (Batting) (team)             360.000 (median  247.000)  -0.21%
  v DPPG (team)                       0.510 (median    0.670)  -0.19%
  v q2 win pct (team)                 0.909 (median    0.364)  -0.16%
  v gs (pitching)                    19.000 (median    1.000)  -0.15%
  v h (pitching)                     91.000 (median   27.000)  -0.15%
  v hr/9 (pitching)                   1.210 (median    1.000)  -0.13%


In [6]:
# The gain fallback works with no SHAP installed
fallback = explain_prediction("Kade Anderson", 2025, method="gain", top_n=3)
print("method:", fallback["method"])
for row in fallback["strengths"]:
    print(f"  ^ {row['label']:28s} {row['impact']:+.2f}")

method: gain
  ^ so (pitching)                +31.93
  ^ ip (pitching)                +9.98
  ^ q1 wins (team)               +8.52


## A whole season, ranked

`draft_board` scores every eligible player. Comparing against `actual_pick`
shows where the model agreed with the draft and where it did not.

In [7]:
board = draft_board(2025, n=15)
print(f"  {'#':>3} {'player':24s} {'team':6s} {'P(draft)':>9s} {'grade':>6s} "
      f"{'proj':>6s} {'actual':>7s}")
for row in board:
    projected = f"{row['predicted_order']:.0f}" if row["predicted_order"] else "-"
    actual = f"#{row['actual_pick']}" if row["actual_pick"] else "undrafted"
    print(f"  {row['rank']:>3} {row['name']:24s} {row['team']:6s} "
          f"{row['draft_probability']:>9.3f} {row['draft_grade']:>6s} "
          f"{projected:>6s} {actual:>7s}")

    # player                   team    P(draft)  grade   proj  actual
    1 Jamie Arnold             FSU        0.995     A+      2     #11
    2 Mitch Voit               MICH       0.994     A+     44     #38
    3 Andrew Fischer           TENN       0.993     A+      1     #20
    4 Blake Gillespie          CLT        0.992     A+     48    #284
    5 Ben Jacobs               ASU        0.991     A+     65     #98
    6 Gavin Kilen              TENN       0.986     A+     17     #13
    7 Jack Martinez            ASU        0.986     A+    177    #243
    8 Anthony Eyanson          LSU        0.986     A+      1     #87
    9 Cade Obermueller         IOWA       0.986     A+     75     #63
   10 Devin Taylor             IU         0.985     A+      8     #48
   11 Kade Anderson            LSU        0.985     A+      1      #3
   12 J.D. Thompson            VAN        0.985     A+      1     #59
   13 Justin Lamkin            TA&M       0.983     A+     43     #71
   14 Cody Bowker   

In [8]:
# How much of the top of the board was actually drafted?
top50 = draft_board(2025, n=50)
hit = sum(1 for row in top50 if row["actual_pick"])
print(f"{hit} of the top 50 were drafted ({hit / 50:.0%})")

# Draft position is suppressed below 25%: the order model is trained only on
# drafted players, so applying it lower down would be extrapolation.
low = [r for r in draft_board(2025, n=2000) if r["draft_probability"] < 0.25]
print(f"\n{len(low)} players below the 25% threshold; "
      f"all have predicted_order suppressed: "
      f"{all(r['predicted_order'] is None for r in low)}")

48 of the top 50 were drafted (96%)



1539 players below the 25% threshold; all have predicted_order suppressed: True


## Scoring a line that is not in the data

Supply as much or as little as you have. Unspecified statistics stay missing,
which the models handle natively, and the result reports how much was imputed.

In [9]:
result = predict_from_stats(
    "pitcher", age=21,
    stats={"era": 2.40, "so": 130, "bb": 25, "ip": 95.0,
           "h": 68, "hr": 5, "g": 16, "gs": 16, "tbf": 370},
    team="LSU", season=2025, name="Prospect A",
)
print(result["report"])

  Prospect A  |  pitcher, age 21  |  2025 context  (LSU)
  Draft grade: A+   (modelled probability 97.0%)
  Projected college draft order: ~64
------------------------------------------------------------------
  Supplied 9 statistics; 63 left unset (85%).
  Confidence: low.
  Team context imputed from the 2025 median (1 fields).


In [10]:
print("supplied:", result["supplied_features"])
print("confidence:", result["confidence"])
print("draft probability:", round(result["draft_probability"], 4))
print("predicted order:", result["predicted_order"])

supplied: ['bb_pitch', 'era_pitch', 'g_pitch', 'gs_pitch', 'h_pitch', 'hr_pitch', 'ip_pitch', 'so_pitch', 'tbf_pitch']
confidence: low
draft probability: 0.9696
predicted order: 63.81031036376953


In [11]:
# A weaker line, with no team given -- context falls back to the season median
weak = predict_from_stats(
    "batter", age=19,
    stats={"avg": 0.240, "hr": 1, "pa": 90, "ab": 80},
    season=2025, name="Prospect B",
)
print(weak["report"])

  Prospect B  |  batter, age 19  |  2025 context  (league median)
  Draft grade: C   (modelled probability 27.0%)
  Projected college draft order: ~235
------------------------------------------------------------------
  Supplied 4 statistics; 68 left unset (92%).
  Confidence: low.
  Team context imputed from the 2025 median (73 fields).


In [12]:
# The model responds to the input: same role and age, different production
strong = predict_from_stats("pitcher", 21,
    {"era": 1.80, "so": 150, "bb": 15, "ip": 100.0, "h": 60, "hr": 3,
     "g": 16, "gs": 16}, team="LSU", season=2025)
poor = predict_from_stats("pitcher", 21,
    {"era": 7.50, "so": 12, "bb": 20, "ip": 18.0, "h": 30, "hr": 6,
     "g": 9, "gs": 1}, team="LSU", season=2025)

print(f"  strong line: {strong['draft_probability']:.1%}  "
      f"grade {strong['draft_grade']}")
print(f"  poor line  : {poor['draft_probability']:.1%}  "
      f"grade {poor['draft_grade']}")

  strong line: 96.3%  grade A+
  poor line  : 38.5%  grade C+


## The model card

Published as a function rather than a documentation footnote, so the
limitations travel with the predictions.

In [13]:
card = model_card()
print(f"version    : {card['model_version']}")
print(f"trained on : {card['train_years']}")
print(f"tested on  : {card['test_year']}")
print(f"eligibility: {card['eligibility']}")

print(f"\nstage 1 (drafted or not): {card['stage1']['metrics']}")
print(f"stage 2 (draft order)   : {card['stage2']['metrics']}")

version    : s1s2-2026.1
trained on : [2021, 2022, 2023, 2024]
tested on  : 2025
eligibility: {'age': 21, 'seasons': 3, 'unknown_treated_as_eligible': False}

stage 1 (drafted or not): {'base_rate': 0.0661, 'n_test': 6478, 'n_train': 18700, 'pr_auc': 0.7077, 'roc_auc': 0.9623}
stage 2 (draft order)   : {'mae': 76.16, 'n_test': 428, 'n_train': 1605, 'p_value': 1.4936201135256453e-51, 'spearman': 0.6442}


In [14]:
print("Stage 3 is deliberately not shipped:")
print(" ", card["stage3"]["reason"])

Stage 3 is deliberately not shipped:
  A bonus/slot ratio model was attempted and scored a rank correlation of 0.003 on held-out data -- indistinguishable from noise. Slot values are published facts and are available via draft_detail_utils.slot_value().


In [15]:
print("Limitations:")
for i, limitation in enumerate(card["limitations"], 1):
    print(f"\n  {i}. {limitation}")

Limitations:

  1. Stage 1 precision depends on the base rate of the population it is applied to. On the held-out season roughly 7% of eligible players were drafted; applied to a pre-screened shortlist, precision is higher, and applied to every player in the country, lower.

  2. Draft eligibility is inferred from seasons completed and from age, and age is itself estimated for most players. is_draft_eligible() returns the basis so the inference is visible.

  3. The order model is trained only on players who were drafted. Applying it below a 25% draft probability is extrapolation, and it is suppressed there.

  4. No third stage. A bonus/slot ratio model scored a rank correlation of 0.003 on held-out data and is not shipped; slot values are published facts, available via draft_detail_utils.slot_value().

  5. Trained on 2021-2024 and tested on 2025, a single held-out season. These are not cross-validated estimates.


In [16]:
reference = card["reference_implementation"]
print("For comparison only -- NOT this package's numbers:")
print(f"  {reference['note']}\n")
print(f"  reference PR-AUC   {reference['stage1_pr_auc']}   "
      f"vs this package {card['stage1']['metrics']['pr_auc']}")
print(f"  reference ROC-AUC  {reference['stage1_roc_auc']}   "
      f"vs this package {card['stage1']['metrics']['roc_auc']}")
print(f"  reference Spearman {reference['stage2_spearman']}   "
      f"vs this package {card['stage2']['metrics']['spearman']}")

For comparison only -- NOT this package's numbers:
  The research implementation this was ported from used proprietary third-party metrics as features. Its numbers are listed for comparison only; they are NOT this package's performance, and its test year was 2026 rather than 2025.

  reference PR-AUC   0.725   vs this package 0.7077
  reference ROC-AUC  0.949   vs this package 0.9623
  reference Spearman 0.653   vs this package 0.6442
